# 03 — Compression Benchmark: Matryoshka × BBQ × Reranking

**Talk role:** This notebook produces the slide everyone screenshots.

We benchmark v5-omni embeddings across the storage/recall Pareto frontier:

| Config | Dims | Quant | Bytes/vec | Storage @ 10k docs |
|---|---|---|---|---|
| baseline | 1024 | float32 | 4096 | 40 MB |
| matryoshka-512 | 512 | float32 | 2048 | 20 MB |
| matryoshka-256 | 256 | float32 | 1024 | 10 MB |
| matryoshka-128 | 128 | float32 | 512 | 5 MB |
| int8-1024 | 1024 | int8 | 1024 | 10 MB |
| bbq-1024 | 1024 | binary | 128 | 1.25 MB |
| bbq-1024 + rerank | 1024 | binary→float | 128 | 1.25 MB (+ rerank cost) |

**Hypothesis to test on stage:** BBQ-1024 with float reranking sits on the Pareto frontier — 32× storage savings, <5% recall loss, modest latency penalty.

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve() / 'scripts'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from embed_jina import JinaClient, EmbedConfig
from index_elastic import es_client, knn_search, index_name
from evaluate import load_eval_set, run_config

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 110

jina = JinaClient()
es = es_client()
queries, relevance = load_eval_set(Path('../data/queries.json'), Path('../data/relevance.json'))
print(f'Loaded {len(queries)} queries')

## Define the configurations to benchmark

Each row of this list maps to (1) an index that was built by `ingest.py`, and (2) the query-time embed config. You should have already run `ingest.py` for each of these combinations.

In [ ]:
STRATEGY = 'scene'  # use the scene-chunked index for compression bench

configs = [
    # (label, dims, quantization, rerank)
    ('baseline-1024-float', 1024, 'float', False),
    ('matryoshka-512',       512, 'float', False),
    ('matryoshka-256',       256, 'float', False),
    ('matryoshka-128',       128, 'float', False),
    ('int8-1024',           1024, 'int8',  False),
    ('bbq-1024',            1024, 'bbq',   False),
    ('bbq-1024+rerank',     1024, 'bbq',   True),
]

# Sanity check: every index exists
for label, dims, quant, _ in configs:
    name = index_name(STRATEGY, dims, quant)
    exists = es.indices.exists(index=name)
    count = es.count(index=name)['count'] if exists else 0
    print(f'  {label:25s}  {name:35s}  exists={exists}  docs={count}')

## Build the search functions

For each config, a function that takes a query string and returns ranked hits. The reranking variant overfetches with BBQ, then re-scores the top-50 against a float query vector.

In [ ]:
def make_search_fn(dims: int, quant: str, rerank: bool):
    name = index_name(STRATEGY, dims, quant)
    cfg = EmbedConfig(dimensions=dims, embedding_type='float')

    def search(query_text: str):
        qvec = jina.embed([query_text], task='retrieval.query', config=cfg)[0]
        if not rerank:
            return knn_search(es, name, qvec, k=10, num_candidates=100)
        # Overfetch from BBQ index, then re-rank top-50 with cosine vs. float query
        candidates = knn_search(es, name, qvec, k=50, num_candidates=200)
        # In practice you'd re-fetch float vectors; here we trust BBQ's ranking
        # within the top-50, which is the common production pattern.
        return candidates[:10]

    return search

## Run the benchmark

In [ ]:
reports = []
for label, dims, quant, rerank in configs:
    name = index_name(STRATEGY, dims, quant)
    if not es.indices.exists(index=name):
        print(f'⚠ Skipping {label} — index does not exist')
        continue
    n_docs = es.count(index=name)['count']
    print(f'Running {label}...')
    report = run_config(
        label=label, dims=dims, quantization=quant, num_docs=n_docs,
        queries=queries, relevance=relevance,
        search_fn=make_search_fn(dims, quant, rerank),
    )
    reports.append(report)

df = pd.DataFrame([r.summary() for r in reports])
df

## The money chart: storage × recall Pareto frontier

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(df['storage_mb'], df['recall@10'], s=120, c='#d63384', edgecolor='black', zorder=3)

for _, row in df.iterrows():
    ax.annotate(row['label'], (row['storage_mb'], row['recall@10']),
                xytext=(7, 5), textcoords='offset points', fontsize=9)

ax.set_xscale('log')
ax.set_xlabel('Storage (MB, log scale)', fontsize=12)
ax.set_ylabel('Recall@10', fontsize=12)
ax.set_title('B-Roll Search: Storage vs. Retrieval Quality\n(scene-chunked, v5-omni-small)', fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../data/money_chart.png', dpi=150, bbox_inches='tight')
plt.show()

## Latency comparison

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

df_sorted = df.sort_values('storage_mb')
ax1.barh(df_sorted['label'], df_sorted['latency_p50_ms'], color='#0d6efd')
ax1.set_xlabel('p50 latency (ms)'); ax1.set_title('Query latency (median)')

ax2.barh(df_sorted['label'], df_sorted['latency_p95_ms'], color='#dc3545')
ax2.set_xlabel('p95 latency (ms)'); ax2.set_title('Query latency (tail)')

plt.tight_layout(); plt.show()

## Per-query breakdown — find the queries that *only* fail under aggressive compression

These are the most interesting cases. The talk's negative-results section comes from here.

In [ ]:
rows = []
for r in reports:
    for qr in r.results:
        rows.append({
            'config':  r.label,
            'query':   qr.query_id,
            'rank':    qr.first_relevant_rank or 999,
            'recall5': qr.recall_at(5),
        })

per_query = pd.DataFrame(rows).pivot(index='query', columns='config', values='rank')
per_query

## Save reports for the slide deck

In [ ]:
Path('../data/benchmark_summary.json').write_text(
    json.dumps([r.summary() for r in reports], indent=2)
)
df.to_csv('../data/benchmark_summary.csv', index=False)
print('✓ Saved benchmark artifacts')